# Chapter 9 - Optimization for machine learning

*This notebook contains all the sample code in Chapter 9.*

## Outline

- [Iterative optimization algorithms](#Iterative)
- [Unconstrained methods](#Unconstrained)
    - [First-order unconstrained methods](#Unconstrained_FO)
    - [Second-order unconstrained methods](#Unconstrained_SO)
    - [Nonsmooth objectives and regularization](#Nonsmooth)
    - [Fixed-point iteration](#FixedPoint)
- [Constrained methods](#Constrained)
    - [Equality constraints and Lagrange multipliers](#Equality)
    - [Algorithms for constrained problems](#AlgoConst)

## Iterative optimization algorithms <a id="Iterative"></a>

Most machine-learning objectives are solved **iteratively*. A broad class of methods can be written as:

$$ \mathbf{w}^{(k+1)} = \mathbf{w}^{(k)} + \mu_k \mathbf{d}^{(k)}, $$

where $\mathbf{d}^{(k)}$ is a search direction and $\mu_k>0$ is the *step length* or *learning rate*.

Two choices must therefore be made at every iteration: the direction in which to move and the distance to travel.

A *fixed* learning rate is simple, but its appropriate value depends on the curvature and scaling of the objective. A line-search method chooses $\mu_k$ at each iteration. One widely used strategy is **Armijo backtracking**. Starting from an initial value $\mu$, it *repeatedly* reduces the step until:

In [1]:
import numpy as np

def backtracking_line_search(fun, grad, x, direction, mu0=1.0, contraction=0.5, c1=1e-4):
    """Return a step satisfying the Armijo condition."""
    g = grad(x)
    slope = np.dot(g, direction)
    if slope >= 0:
        raise ValueError("direction must be a descent direction")

    mu = mu0
    fx = fun(x)
    while fun(x + mu * direction) > fx + c1 * mu * slope:
        mu *= contraction
        if mu < 1e-16:
            break
    return mu

## Unconstrained methods <a id="Unconstrained"></a>

### First-order unconstrained methods <a id="Unconstrained_FO"></a>

#### Gradient descent

**Gradient descent** selects:
$$ \mathbf{d}^{(k)}=-\nabla f(\mathbf{w}^{(k)}), $$
and therefore performs the update:
$$ \mathbf{w}^{(k+1)} = \mathbf{w}^{(k)} - \mu_k \nabla f(\mathbf{w}^{(k)}). $$


In [2]:
def gradient_descent(fun, grad, x0, learning_rate=0.1, max_iter=1000,
                     tol=1e-8, use_line_search=False):
    """Minimize a differentiable function by gradient descent."""
    x = np.asarray(x0, dtype=float).copy()
    history = [fun(x)]

    for _ in range(max_iter):
        g = grad(x)
        if np.linalg.norm(g) <= tol:
            break

        direction = -g
        if use_line_search:
            step = backtracking_line_search(fun, grad, x, direction)
        else:
            step = learning_rate

        x_new = x + step * direction
        history.append(fun(x_new))

        if np.linalg.norm(x_new - x) <= tol * (1.0 + np.linalg.norm(x)):
            x = x_new
            break
        x = x_new

    return x, np.asarray(history)

#### Batch, stochastic, and mini-batch gradients

*Full-batch* gradient descent uses all $N$ samples at each iteration. **Stochastic gradient** descent (SGD) uses one randomly selected sample, whereas *mini-batch* gradient descent uses a subset $\mathcal{B}_k$.

In [3]:
def mini_batch_sgd(batch_gradient, x0, n_samples, learning_rate=0.01,
                   batch_size=32, epochs=100, random_state=0):
    """Mini-batch SGD with reshuffling at every epoch."""
    rng = np.random.default_rng(random_state)
    x = np.asarray(x0, dtype=float).copy()

    for _ in range(epochs):
        indices = rng.permutation(n_samples)
        for start in range(0, n_samples, batch_size):
            batch = indices[start:start + batch_size]
            x -= learning_rate * batch_gradient(x, batch)

    return x

#### Momentum and Nesterov acceleration

**Momentum** maintains a velocity vector that averages information from previous gradients:
$$
\mathbf{v}^{(k+1)} =\alpha\mathbf{v}^{(k)}-\mu\nabla f(\mathbf{w}^{(k)}),
$$
$$
\mathbf{w}^{(k+1)} =\mathbf{w}^{(k)}+\mathbf{v}^{(k+1)},
$$
where $0\leq\alpha<1$. Momentum can accelerate progress in directions where successive gradients are consistent and reduce oscillation in directions of high curvature.

**Nesterov accelerated gradient** evaluates the gradient at a look-ahead point:
$$
\widetilde{\mathbf{w}}^{(k)} = \mathbf{w}^{(k)}+\alpha\mathbf{v}^{(k)},
$$
$$
\mathbf{v}^{(k+1)} = \alpha\mathbf{v}^{(k)}-\mu\nabla f(\widetilde{\mathbf{w}}^{(k)}),
$$
$$
\mathbf{w}^{(k+1)} = \mathbf{w}^{(k)}+\mathbf{v}^{(k+1)}.
$$

In [4]:
def momentum_descent(fun, grad, x0, learning_rate=0.01, alpha=0.9,
                     max_iter=1000, tol=1e-8, nesterov=False):
    """Gradient descent with classical or Nesterov momentum."""
    x = np.asarray(x0, dtype=float).copy()
    velocity = np.zeros_like(x)
    history = [fun(x)]

    for _ in range(max_iter):
        point = x + alpha * velocity if nesterov else x
        g = grad(point)
        velocity = alpha * velocity - learning_rate * g
        x_new = x + velocity
        history.append(fun(x_new))

        if np.linalg.norm(x_new - x) <= tol * (1.0 + np.linalg.norm(x)):
            x = x_new
            break
        x = x_new

    return x, np.asarray(history)

#### Adaptive learning-rate methods

**AdaGrad** accumulates squared gradients component-wise:
$$
\mathbf{r}^{(k)} = \mathbf{r}^{(k-1)}+\mathbf{g}^{(k)}\odot\mathbf{g}^{(k)},
$$
$$
\mathbf{w}^{(k+1)} = \mathbf{w}^{(k)}-\mu\frac{\mathbf{g}^{(k)}}{\sqrt{\mathbf{r}^{(k)}}+\varepsilon}.
$$
It is effective for sparse gradients, but its learning rates decrease monotonically and may become excessively small.

In [5]:
def adagrad(fun, grad, x0, learning_rate=0.1, max_iter=1000, tol=1e-8, epsilon=1e-8):
    x = np.asarray(x0, dtype=float).copy()
    accumulator = np.zeros_like(x)
    history = [fun(x)]

    for _ in range(max_iter):
        g = grad(x)
        accumulator += g * g
        x_new = x - learning_rate * g / (np.sqrt(accumulator) + epsilon)
        history.append(fun(x_new))
        if np.linalg.norm(x_new - x) <= tol * (1.0 + np.linalg.norm(x)):
            x = x_new
            break
        x = x_new
    return x, np.asarray(history)

**RMSProp** replaces the cumulative sum with an exponentially weighted average:
$$
\mathbf{r}^{(k)} = \rho\mathbf{r}^{(k-1)}+(1-\rho)\mathbf{g}^{(k)}\odot\mathbf{g}^{(k)},
$$
$$
\mathbf{w}^{(k+1)} = \mathbf{w}^{(k)}-\mu\frac{\mathbf{g}^{(k)}}{\sqrt{\mathbf{r}^{(k)}}+\varepsilon}.
$$

In [6]:
def rmsprop(fun, grad, x0, learning_rate=0.01, beta=0.9,
            max_iter=1000, tol=1e-8, epsilon=1e-8):
    x = np.asarray(x0, dtype=float).copy()
    second_moment = np.zeros_like(x)
    history = [fun(x)]

    for _ in range(max_iter):
        g = grad(x)
        second_moment = beta * second_moment + (1.0 - beta) * g * g
        x_new = x - learning_rate * g / (np.sqrt(second_moment) + epsilon)
        history.append(fun(x_new))
        if np.linalg.norm(x_new - x) <= tol * (1.0 + np.linalg.norm(x)):
            x = x_new
            break
        x = x_new
    return x, np.asarray(history)

**Adam** combines estimates of the first and second moments of the gradient:
$$
\mathbf{m}^{(k)} = \beta_1\mathbf{m}^{(k-1)}+(1-\beta_1)\mathbf{g}^{(k)},
$$
$$
\mathbf{v}^{(k)} = \beta_2\mathbf{v}^{(k-1)}+(1-\beta_2)\mathbf{g}^{(k)}\odot\mathbf{g}^{(k)}.
$$
Because both moments are initialized at zero, Adam uses the bias-corrected quantities:
$$
\widehat{\mathbf{m}}^{(k)}=
\frac{\mathbf{m}^{(k)}}{1-\beta_1^k},
\qquad
\widehat{\mathbf{v}}^{(k)}=
\frac{\mathbf{v}^{(k)}}{1-\beta_2^k},
$$
and updates:
$$
\mathbf{w}^{(k+1)}=\mathbf{w}^{(k)}-
\mu\frac{\widehat{\mathbf{m}}^{(k)}}
{\sqrt{\widehat{\mathbf{v}}^{(k)}}+\varepsilon}.
$$

In [7]:
def adam(fun, grad, x0, learning_rate=0.01, beta1=0.9, beta2=0.999,
         max_iter=1000, tol=1e-8, epsilon=1e-8):
    x = np.asarray(x0, dtype=float).copy()
    first_moment = np.zeros_like(x)
    second_moment = np.zeros_like(x)
    history = [fun(x)]

    for k in range(1, max_iter + 1):
        g = grad(x)
        first_moment = beta1 * first_moment + (1.0 - beta1) * g
        second_moment = beta2 * second_moment + (1.0 - beta2) * g * g
        m_hat = first_moment / (1.0 - beta1 ** k)
        v_hat = second_moment / (1.0 - beta2 ** k)
        x_new = x - learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)
        history.append(fun(x_new))

        if np.linalg.norm(x_new - x) <= tol * (1.0 + np.linalg.norm(x)):
            x = x_new
            break
        x = x_new

    return x, np.asarray(history)

### Second-order unconstrained methods <a id="Unconstrained_SO"></a>

#### Newton's method

A second-order *Taylor approximation* around $\mathbf{w}^{(k)}$ is:
$$
f(\mathbf{w}^{(k)}+\mathbf{d})\approx
f(\mathbf{w}^{(k)})+
\nabla f(\mathbf{w}^{(k)})^\top\mathbf{d}
+\frac{1}{2}\mathbf{d}^\top
\nabla^2f(\mathbf{w}^{(k)})\mathbf{d}.
$$
Minimizing this quadratic model gives the Newton direction:
$$
\nabla^2f(\mathbf{w}^{(k)})\mathbf{d}^{(k)} = -\nabla f(\mathbf{w}^{(k)})
\quad\longrightarrow\quad
\mathbf{H}_k \mathbf{d}^{(k)} = - \mathbf{g}^{(k)}.
$$
The update is then:
$$
\mathbf{w}^{(k+1)} = \mathbf{w}^{(k)} + \mu_k \mathbf{d}^{(k)}.
$$
It is preferable to solve the linear system \eqref{eq:Newton} rather than explicitly compute the inverse Hessian ($\mathbf{d}^{(k)} = - \mathbf{H}_k^{-1} \mathbf{g}^{(k)}$). Near a strict minimizer, Newton's method can converge quadratically. Far from the solution, however, an indefinite Hessian may produce a non-descent direction. Damping and line search improve robustness.

In [8]:
def newton_method(fun, grad, hessian, x0, max_iter=100,
                  tol=1e-10, damping=1e-8):
    """Damped Newton method with Armijo backtracking."""
    x = np.asarray(x0, dtype=float).copy()
    history = [fun(x)]

    for _ in range(max_iter):
        g = grad(x)
        if np.linalg.norm(g) <= tol:
            break

        h = hessian(x) + damping * np.eye(x.size)
        try:
            direction = np.linalg.solve(h, -g)
        except np.linalg.LinAlgError:
            direction = -g

        if np.dot(g, direction) >= 0:
            direction = -g

        step = backtracking_line_search(fun, grad, x, direction)
        x_new = x + step * direction
        history.append(fun(x_new))

        if np.linalg.norm(x_new - x) <= tol * (1.0 + np.linalg.norm(x)):
            x = x_new
            break
        x = x_new

    return x, np.asarray(history)

For nonlinear least-squares problems, the **Levenberg-Marquardt** method uses a damped Gauss--Newton system. The related update:
$$
\left(\mathbf{H}_k+\lambda_k\mathbf{I}\right) \mathbf{d}^{(k)}=-\mathbf{g}^{(k)},
$$
is more generally called a damped or regularized Newton step; it should not always be identified with Levenberg--Marquardt, whose structure specifically exploits a sum-of-squares objective.

#### Quasi-Newton methods and BFGS

**Quasi-Newton** methods avoid computing the exact Hessian. BFGS maintains an approximation $\mathbf{B}_k$ of the inverse Hessian. Define:
$$
\mathbf{s}_k=\mathbf{w}^{(k+1)}-\mathbf{w}^{(k)},
\qquad
\mathbf{y}_k=\nabla f(\mathbf{w}^{(k+1)})-
\nabla f(\mathbf{w}^{(k)}),
$$
and $\rho_k=(\mathbf{y}_k^\top\mathbf{s}_k)^{-1}$. The inverse-BFGS update is:
$$
\mathbf{B}_{k+1}=
\left(\mathbf{I}-\rho_k\mathbf{s}_k\mathbf{y}_k^\top\right)
\mathbf{B}_k
\left(\mathbf{I}-\rho_k\mathbf{y}_k\mathbf{s}_k^\top\right)
+\rho_k\mathbf{s}_k\mathbf{s}_k^\top.
$$
When $\mathbf{y}_k^\top\mathbf{s}_k>0$, positive definiteness is preserved. The search direction is $-\mathbf{B}_k\nabla f(\mathbf{w}^{(k)})$.

In [9]:
def bfgs(fun, grad, x0, max_iter=500, tol=1e-8):
    """BFGS using an inverse-Hessian approximation."""
    x = np.asarray(x0, dtype=float).copy()
    inverse_hessian = np.eye(x.size)
    g = grad(x)
    history = [fun(x)]

    for _ in range(max_iter):
        if np.linalg.norm(g) <= tol:
            break

        direction = -inverse_hessian @ g
        if np.dot(g, direction) >= 0:
            direction = -g
            inverse_hessian = np.eye(x.size)

        step = backtracking_line_search(fun, grad, x, direction)
        x_new = x + step * direction
        g_new = grad(x_new)

        s = x_new - x
        y = g_new - g
        ys = np.dot(y, s)

        if ys > 1e-12:
            rho = 1.0 / ys
            identity = np.eye(x.size)
            left = identity - rho * np.outer(s, y)
            inverse_hessian = (
                left @ inverse_hessian @ left.T
                + rho * np.outer(s, s)
            )
        else:
            inverse_hessian = np.eye(x.size)

        x, g = x_new, g_new
        history.append(fun(x))

    return x, np.asarray(history)

#### Conjugate gradient for quadratic objectives

Consider the strictly convex quadratic objective:
$$
f(\mathbf{w})=\frac{1}{2}\mathbf{w}^\top
\mathbf{Q}\mathbf{w}-\mathbf{b}^\top\mathbf{w},
$$
where $\mathbf{Q}$ is symmetric positive definite. Its minimizer solves:
$$
\mathbf{Q}\mathbf{w}^{\ast}=\mathbf{b}.
$$
The **conjugate-gradient** method constructs $\mathbf{Q}$-conjugate search directions and, in exact arithmetic, reaches the solution in at most $n$ iterations. It requires only matrix-vector products and is therefore attractive for large sparse systems.

In [10]:
def conjugate_gradient(matrix, vector, x0=None, tol=1e-10, max_iter=None):
    """Solve A x = b for a symmetric positive-definite A."""
    matrix = np.asarray(matrix, dtype=float)
    vector = np.asarray(vector, dtype=float)
    n = vector.size
    x = np.zeros(n) if x0 is None else np.asarray(x0, dtype=float).copy()

    if max_iter is None:
        max_iter = n

    residual = vector - matrix @ x
    direction = residual.copy()
    residual_norm_sq = np.dot(residual, residual)

    for _ in range(max_iter):
        product = matrix @ direction
        denominator = np.dot(direction, product)
        if denominator <= 0:
            raise ValueError("matrix must be symmetric positive definite")

        alpha = residual_norm_sq / denominator
        x += alpha * direction
        residual -= alpha * product
        new_norm_sq = np.dot(residual, residual)

        if np.sqrt(new_norm_sq) <= tol:
            break

        beta = new_norm_sq / residual_norm_sq
        direction = residual + beta * direction
        residual_norm_sq = new_norm_sq

    return x

#### Example: the Rosenbrock function

The **Rosenbrock function** is a standard nonconvex test problem:
$$
f(w_1,w_2)=(1-w_1)^2+100(w_2-w_1^2)^2.
$$
Its global minimizer is $(1,1)$. The following code compares several methods implemented above.

In [11]:
def rosenbrock(x):
    return (1.0 - x[0]) ** 2 + 100.0 * (x[1] - x[0] ** 2) ** 2

def rosenbrock_gradient(x):
    return np.array([
        -2.0 * (1.0 - x[0]) - 400.0 * x[0] * (x[1] - x[0] ** 2),
        200.0 * (x[1] - x[0] ** 2)
    ])

def rosenbrock_hessian(x):
    return np.array([
        [2.0 - 400.0 * x[1] + 1200.0 * x[0] ** 2, -400.0 * x[0]],
        [-400.0 * x[0], 200.0]
    ])

x0 = np.array([-1.2, 1.0])

x_gd, _ = gradient_descent(
    rosenbrock, rosenbrock_gradient, x0,
    max_iter=20000, use_line_search=True
)
x_adam, _ = adam(
    rosenbrock, rosenbrock_gradient, x0,
    learning_rate=0.01, max_iter=20000
)
x_newton, _ = newton_method(
    rosenbrock, rosenbrock_gradient,
    rosenbrock_hessian, x0
)
x_bfgs, _ = bfgs(rosenbrock, rosenbrock_gradient, x0)

print("Gradient descent:", x_gd)
print("Adam:", x_adam)
print("Newton:", x_newton)
print("BFGS:", x_bfgs)

Gradient descent: [0.9999906  0.99998113]
Adam: [0.99999856 0.99999711]
Newton: [1. 1.]
BFGS: [1.         0.99999999]


### Nonsmooth objectives and regularization <a id="Nonsmooth"></a>

Regularization modifies the training objective to control model complexity or improve numerical conditioning:
$$
\widetilde{f}(\mathbf{w}) = f(\mathbf{w})+\lambda R(\mathbf{w}),
$$
where $\lambda\geq0$. 

The $\ell_2$ penalty shrinks coefficients continuously. The $\ell_1$ penalty is nondifferentiable at zero and can produce exact zeros, thereby encouraging sparse solutions.


Ordinary gradient descent cannot directly use the derivative of $\|\mathbf{w}\|_1$ at zero. A **proximal-gradient** method instead separates a differentiable function $f$ from a possibly *nonsmooth* regularizer $R$:
$$
\mathbf{w}^{(k+1)}=
\operatorname{prox}_{\mu\lambda R}
\left(\mathbf{w}^{(k)}-
\mu\nabla f(\mathbf{w}^{(k)})\right).
$$
For the $\ell_1$ norm, the proximal operator $\operatorname{prox}$ is compute by the soft thresholding:
$$
\mathcal{S}_{\tau}(z) = \operatorname{sign}(z) \max\{|z|-\tau, 0\}.
$$

In [12]:
def soft_threshold(z, threshold):
    return np.sign(z) * np.maximum(np.abs(z) - threshold, 0.0)


def proximal_gradient(grad_smooth, x0, regularization, step_size,
                      max_iter=1000, tol=1e-8):
    """Minimize f(x) + regularization * ||x||_1."""
    x = np.asarray(x0, dtype=float).copy()

    for _ in range(max_iter):
        candidate = x - step_size * grad_smooth(x)
        x_new = soft_threshold(candidate, step_size * regularization)

        if np.linalg.norm(x_new - x) <= tol * (1.0 + np.linalg.norm(x)):
            x = x_new
            break
        x = x_new

    return x

### Fixed-point iteration <a id="FixedPoint"></a>

A **fixed point** $\overline{\mathbf{w}}$ of a mapping $\mathbf{T}$ satisfies: $\mathbf{T}(\overline{\mathbf{w}})=\overline{\mathbf{w}}$. The iteration:
$$
\mathbf{w}^{(k+1)}=\mathbf{T}(\mathbf{w}^{(k)}),
$$
converges to a unique fixed point when $\mathbf{T}$ is a contraction on a closed set.


Optimization algorithms can often be interpreted as **fixed-point** methods. For example, a point is stationary for gradient descent precisely when:
$$
\mathbf{w} = \mathbf{w} - \mu \nabla f(\mathbf{w}).
$$
This viewpoint is useful for convergence analysis, but an arbitrary rearrangement of an equation does not necessarily yield a contractive iteration.

In [13]:
def fixed_point(mapping, x0, max_iter=1000, tol=1e-10):
    x = np.asarray(x0, dtype=float).copy()
    for _ in range(max_iter):
        x_new = np.asarray(mapping(x), dtype=float)
        if np.linalg.norm(x_new - x) <= tol * (1.0 + np.linalg.norm(x)):
            return x_new
        x = x_new
    
    return x

## Constrained methods <a id="Constrained"></a>

A **constrained problem** restricts the solution to a feasible set. It can have equality and/or inequality constraints

### Equality constraints and Lagrange multipliers <a id="Equality"></a>

For the equality-constrained quadratic program:
$$
\min_{\mathbf{w}}\quad \frac{1}{2}\mathbf{w}^{\top}\mathbf{Q}\mathbf{w}+\mathbf{c}^{\top}\mathbf{w},
$$
$$
\text{subject to}\quad \mathbf{A}\mathbf{w}=\mathbf{b},
$$

In [14]:
def equality_qp(matrix, linear_term, constraint_matrix, constraint_vector):
    """Solve an equality-constrained convex quadratic program."""
    q = np.asarray(matrix, dtype=float)
    c = np.asarray(linear_term, dtype=float)
    a = np.asarray(constraint_matrix, dtype=float)
    b = np.asarray(constraint_vector, dtype=float)

    zeros = np.zeros((a.shape[0], a.shape[0]))
    kkt = np.block([[q, a.T],
                    [a, zeros]])
    rhs = np.concatenate([-c, b])
    solution = np.linalg.solve(kkt, rhs)

    n = q.shape[0]
		
    return solution[:n], solution[n:]

### Algorithms for constrained problems <a id="AlgoConst"></a>

#### Projected gradient descent

If **projection** onto a closed convex feasible set $\Omega$ is inexpensive, projected gradient descent performs:
$$
\mathbf{w}^{(k+1)}=
\Pi_{\Omega}\!\left(
\mathbf{w}^{(k)}-
\alpha_k\nabla f(\mathbf{w}^{(k)})
\right),
$$
where:
$$
\Pi_{\Omega}(\mathbf{z})=
\mathop{\mathrm{arg\,min}}_{\mathbf{u}\in\Omega}
\left\|\mathbf{u}-\mathbf{z}\right\|_2,
$$
is the Euclidean projection. Projection onto a box constraint $\boldsymbol{\ell}\leq\mathbf{w}\leq\mathbf{u}$ is simply componentwise clipping.

In [15]:
def project_box(x, lower, upper):
    return np.clip(x, lower, upper)


def projected_gradient(fun, grad, projection, x0, step_size=0.1,
                       max_iter=1000, tol=1e-8):
    """Projected gradient descent for a convex feasible set."""
    x = projection(np.asarray(x0, dtype=float))
    history = [fun(x)]

    for _ in range(max_iter):
        x_new = projection(x - step_size * grad(x))
        history.append(fun(x_new))

        if np.linalg.norm(x_new - x) <= tol * (1.0 + np.linalg.norm(x)):
            x = x_new
            break
        x = x_new

    return x, np.asarray(history)

For example, the projection function for $0\leq w_i\leq1$ can be defined as:

In [16]:
projection = lambda x: project_box(
    x,
    lower=np.zeros_like(x),
    upper=np.ones_like(x)
)

#### Quadratic penalty method

**Penalty methods** replace the constrained problem with a sequence of unconstrained problems. A common quadratic penalty is:
$$
F_{\rho}(\mathbf{w})=f(\mathbf{w})+
\frac{\rho}{2}\sum_{i=1}^{P}h_i(\mathbf{w})^2+
\frac{\rho}{2}\sum_{j=1}^{Q}
\max\{0,g_j(\mathbf{w})\}^2.
$$
The coefficient $\rho$ is increased across outer iterations. Larger values enforce feasibility more strongly, but can make the unconstrained problem poorly conditioned. Therefore, an excessively large penalty should not be used from the beginning.

In [17]:
def quadratic_penalty(fun, grad, equality, equality_jacobian,
                      inequality, inequality_jacobian, x0,
                      rho0=1.0, rho_growth=10.0, step_size=0.01,
                      outer_iter=6, inner_iter=2000):
    """Quadratic penalty for h(x)=0 and g(x)<=0."""
    x = np.asarray(x0, dtype=float).copy()
    rho = rho0

    for _ in range(outer_iter):
        def penalized(z):
            h = equality(z)
            g = inequality(z)
            positive = np.maximum(g, 0.0)
            return fun(z) + 0.5 * rho * (
                np.dot(h, h) + np.dot(positive, positive)
            )

        def penalized_grad(z):
            h = equality(z)
            g = inequality(z)
            positive = np.maximum(g, 0.0)
            return (
                grad(z)
                + rho * equality_jacobian(z).T @ h
                + rho * inequality_jacobian(z).T @ positive
            )

        x, _ = gradient_descent(
            penalized, penalized_grad, x, learning_rate=step_size,
            max_iter=inner_iter, tol=1e-9
        )
        rho *= rho_growth

    return x

#### Augmented Lagrangian method

For equality constraints, the **augmented Lagrangian** combines multipliers and a quadratic penalty:
$$
\mathcal{L}_{\rho}(\mathbf{w},\boldsymbol{\lambda})
=f(\mathbf{w})+
\boldsymbol{\lambda}^{\top}\mathbf{h}(\mathbf{w})+
\frac{\rho}{2}\left\|\mathbf{h}(\mathbf{w})\right\|_2^2.
$$
After approximately minimizing with respect to $\mathbf{w}$, the multipliers are updated as:
$$
\boldsymbol{\lambda}^{(k+1)}=
\boldsymbol{\lambda}^{(k)}+
\rho\mathbf{h}(\mathbf{w}^{(k+1)}).
$$
Compared with a pure penalty method, the augmented Lagrangian can enforce constraints accurately without requiring an extremely large penalty coefficient.

In [18]:
def augmented_lagrangian_equality(fun, grad, equality, equality_jacobian, x0,
        rho=10.0, outer_iter=20, step_size=0.01, inner_iter=1000, tol=1e-8):
    """Augmented Lagrangian method for equality constraints."""
    x = np.asarray(x0, dtype=float).copy()
    multiplier = np.zeros_like(equality(x), dtype=float)

    for _ in range(outer_iter):
        def augmented(z):
            h = equality(z)
            return fun(z) + np.dot(multiplier, h) + 0.5 * rho * np.dot(h, h)

        def augmented_grad(z):
            h = equality(z)
            return grad(z) + equality_jacobian(z).T @ (multiplier + rho * h)

        x, _ = gradient_descent(
            augmented, augmented_grad, x, learning_rate=step_size,
            max_iter=inner_iter, tol=1e-10
        )

        residual = equality(x)
        multiplier = multiplier + rho * residual
        if np.linalg.norm(residual) <= tol:
            break

    return x, multiplier

#### Example: equality-constrained optimization

Consider the problem:
$$
\min_{w_1,w_2} (w_1-2)^2+(w_2-2)^2 \qquad \text{subject to} \qquad w_1+w_2=1.
$$
The solution is $\mathbf{w}^{\ast}=[0.5,0.5]^\top$. It can be obtained either by the KKT linear system or by the augmented-Lagrangian implementation.

In [19]:
def objective(x):
    return (x[0] - 2.0) ** 2 + (x[1] - 2.0) ** 2

def objective_gradient(x):
    return 2.0 * (x - np.array([2.0, 2.0]))

def equality(x):
    return np.array([x[0] + x[1] - 1.0])

def equality_jacobian(x):
    return np.array([[1.0, 1.0]])

solution, multiplier = augmented_lagrangian_equality(
    objective, objective_gradient,
    equality, equality_jacobian,
    x0=np.zeros(2), rho=5.0, step_size=0.02
)

print(solution)       # approximately [0.5, 0.5]
print(equality(solution))

[0.5 0.5]
[8.3444287e-09]


No optimization method is uniformly best. The appropriate choice depends on the objective, the number of parameters, the amount of data, and the available derivatives.